In [1]:
# ==============================================================================
# NOTEBOOK 03: PIPELINE C - 1D-CNN+GRU SU VETTORI DI FEATURE
# ==============================================================================
# OBIETTIVO:
# Addestrare e valutare un modello 1D-CNN+GRU utilizzando un vettore combinato
# di feature audio (centroid, rolloff, zcr, chroma, contrast).
# Questo approccio è alternativo a quello basato su spettrogrammi 2D.
# ==============================================================================

# --- 1. SETUP E CONFIGURAZIONE (CORRETTO PER GOOGLE COLAB) ---
print("--- [FASE 1/6] SETUP E CONFIGURAZIONE ---")

import os
import sys
import json
import glob
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from google.colab import drive

import tensorflow as tf
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

# 1. Monta Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Definisci il percorso radice del progetto e aggiungilo al path di sistema
#    Questo blocco cerca un percorso valido per permettere la collaborazione.
known_paths = [
    '/content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab', # Esempio percorso 1
    '/content/drive/MyDrive/Colab/urbanSmartSound_Colab'   # Esempio percorso 2
]
PROJECT_ROOT = None
for path in known_paths:
    if os.path.exists(path):
        PROJECT_ROOT = path
        break

if not PROJECT_ROOT:
    raise FileNotFoundError("ERRORE: Nessuno dei percorsi di progetto conosciuti è stato trovato. Aggiorna 'known_paths' con il tuo percorso.")

print(f"Cartella di progetto trovata a: {PROJECT_ROOT}")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 3. Importa i moduli custom dal progetto
from src import data_loader, models, evaluation

# 4. Configurazione specifica della pipeline
PIPELINE_NAME = "Pipeline_C_Vector"
FEATURE_LOADER_FN = data_loader.load_vector_features  # Usa la funzione per caricare vettori 1D
MODEL_CREATOR_FN = models.create_1d_cnn_gru_model     # Usa il modello 1D corrispondente
MODEL_FILENAME = "model_C_vector.keras"
METADATA_FILENAME = "model_C_vector_metadata.json"

# 5. Definizione dei percorsi assoluti per dati e modelli
FEATURES_DIR = os.path.join(PROJECT_ROOT, "data/processed/features_v2")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/UrbanSound8K.csv")
os.makedirs(MODELS_DIR, exist_ok=True)

print("\nConfigurazione completata:")
print(f"  - Pipeline: {PIPELINE_NAME}")
print(f"  - Funzione di caricamento: {FEATURE_LOADER_FN.__name__}")
print(f"  - Funzione di creazione modello: {MODEL_CREATOR_FN.__name__}")


# --- 2. CARICAMENTO DATI ---
print("\n--- [FASE 2/6] CARICAMENTO DATI ---")
all_data = {}
fold_dirs = sorted(glob.glob(os.path.join(FEATURES_DIR, "fold*")))

if not fold_dirs:
    raise FileNotFoundError(f"Nessuna cartella 'fold*' trovata in {FEATURES_DIR}. Esegui prima il notebook di estrazione feature.")

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)
    print(f"Caricando {fold_name}...")
    X_fold, y_fold = data_loader.collect_fold_data(
        fold_dir,
        FEATURE_LOADER_FN # Non servono altri argomenti per questa funzione
    )
    all_data[fold_name] = (X_fold, y_fold)
print("Caricamento di tutti i dati completato.")



--- [FASE 1/6] SETUP E CONFIGURAZIONE ---
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 1070461045341691065
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 14619377664
locality {
  bus_id: 1
  links {
  }
}
incarnation: 18278261326535154805
physical_device_desc: "device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5"
xla_global_id: 416903419
]
Mounted at /content/drive
Cartella di progetto trovata a: /content/drive/MyDrive/Colab/urbanSmartSound_Colab

Configurazione completata:
  - Pipeline: Pipeline_C_Vector
  - Funzione di caricamento: load_vector_features
  - Funzione di creazione modello: create_1d_cnn_gru_model

--- [FASE 2/6] CARICAMENTO DATI ---
Caricando fold1...


KeyboardInterrupt: 

In [ ]:

# --- 3. CROSS-VALIDATION ---
print("\n--- [FASE 3/6] AVVIO CROSS-VALIDATION ---")

# Carica i nomi delle classi e definisce le dimensioni di input/output
class_names = data_loader.get_class_map(CSV_PATH)
num_classes = len(class_names)
input_shape = list(all_data.values())[0][0][0].shape
print(f"Input shape per il modello: {input_shape}")
print(f"Numero di classi: {num_classes}")

fold_accuracies = []
all_y_true_cv, all_y_pred_cv = [], []

for i, val_fold_name in enumerate(sorted(all_data.keys())):
    print(f"\n--- CV Fold {i+1}/{len(all_data)} (Validation: {val_fold_name}) ---")

    X_val, y_val = all_data[val_fold_name]
    train_folds = [data for name, data in all_data.items() if name != val_fold_name]
    X_train = np.vstack([f[0] for f in train_folds])
    y_train = np.concatenate([f[1] for f in train_folds])

    model = MODEL_CREATOR_FN(input_shape, num_classes)
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        #callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )

    _, acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(acc)
    y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    all_y_true_cv.extend(y_val)
    all_y_pred_cv.extend(y_pred)
    print(f"Accuracy del fold: {acc:.4f}")

mean_acc_cv = np.mean(fold_accuracies)
std_acc_cv = np.std(fold_accuracies)
print(f"\nRisultato Cross-Validation: Accuracy media = {mean_acc_cv:.4f} ± {std_acc_cv:.4f}")




--- [FASE 3/6] AVVIO CROSS-VALIDATION ---
Input shape per il modello: (173, 22)
Numero di classi: 10

--- CV Fold 1/10 (Validation: fold1) ---
Accuracy del fold: 0.6119

--- CV Fold 2/10 (Validation: fold10) ---
Accuracy del fold: 0.5086

--- CV Fold 3/10 (Validation: fold2) ---
Accuracy del fold: 0.5791

--- CV Fold 4/10 (Validation: fold3) ---
Accuracy del fold: 0.5854

--- CV Fold 5/10 (Validation: fold4) ---
Accuracy del fold: 0.5680

--- CV Fold 6/10 (Validation: fold5) ---
Accuracy del fold: 0.6538

--- CV Fold 7/10 (Validation: fold6) ---
Accuracy del fold: 0.5458

--- CV Fold 8/10 (Validation: fold7) ---
Accuracy del fold: 0.6574

--- CV Fold 9/10 (Validation: fold8) ---
Accuracy del fold: 0.5894

--- CV Fold 10/10 (Validation: fold9) ---
Accuracy del fold: 0.6411

Risultato Cross-Validation: Accuracy media = 0.5941 ± 0.0455


In [ ]:

# --- 4. ADDESTRAMENTO MODELLO FINALE ---
print("\n--- [FASE 4/6] ADDESTRAMENTO MODELLO FINALE ---")

X_train_final, y_train_final, X_test_final, y_test_final = data_loader.get_train_test_split_from_folds(
    all_data,
    meta_file_path=CSV_PATH,  # Passiamo il percorso corretto!
    test_size=0.2             # Questo è lo split 80/20 che volevi
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_final), y=y_train_final)
class_weight_dict = dict(enumerate(class_weights))
print("Pesi delle classi calcolati per gestire lo sbilanciamento.")

final_model = MODEL_CREATOR_FN(input_shape, num_classes)
print("Avvio dell'addestramento del modello finale...")

final_history = final_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_test_final, y_test_final),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        #EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=7, min_lr=1e-6)
    ],
    verbose=1
)



--- [FASE 4/6] ADDESTRAMENTO MODELLO FINALE ---
Pesi delle classi calcolati per gestire lo sbilanciamento.
Avvio dell'addestramento del modello finale...
Epoch 1/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.2815 - loss: 1.9940 - val_accuracy: 0.3648 - val_loss: 1.7480 - learning_rate: 5.0000e-04
Epoch 2/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.3995 - loss: 1.6647 - val_accuracy: 0.4060 - val_loss: 1.6365 - learning_rate: 5.0000e-04
Epoch 3/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4181 - loss: 1.5786 - val_accuracy: 0.4249 - val_loss: 1.6215 - learning_rate: 5.0000e-04
Epoch 4/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.4554 - loss: 1.4895 - val_accuracy: 0.4572 - val_loss: 1.5701 - learning_rate: 5.0000e-04
Epoch 5/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4789 - loss: 1.3761 - val_accuracy: 0.4410 - val_loss: 1.6929 - learning_rate: 5.0000e-04
Epoch 6/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step -

In [ ]:
# --- 5. VALUTAZIONE MODELLO FINALE ---
print("\n--- [FASE 5/6] VALUTAZIONE MODELLO FINALE ---")

final_loss, final_accuracy = final_model.evaluate(X_test_final, y_test_final, verbose=0)
print("\nPerformance finale sul Test Set:")
print(f"  - Loss: {final_loss:.4f}")
print(f"  - Accuracy: {final_accuracy:.4f}")


# --- 6. SALVATAGGIO MODELLO E METADATI ---
print("\n--- [FASE 6/6] SALVATAGGIO MODELLO E METADATI ---")

model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)
final_model.save(model_path)
print(f"Modello salvato in: {model_path}")

metadata = {
    "pipeline_name": PIPELINE_NAME,
    "feature_loader": FEATURE_LOADER_FN.__name__,
    "model_creator": MODEL_CREATOR_FN.__name__,
    "model_filename": MODEL_FILENAME,
    "input_shape": input_shape,
    "num_classes": num_classes,
    "class_names": class_names,
    "cv_performance": {"mean_accuracy": mean_acc_cv, "std_accuracy": std_acc_cv},
    "final_test_performance": {"accuracy": final_accuracy, "loss": final_loss}
}
metadata_path = os.path.join(MODELS_DIR, METADATA_FILENAME)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadati salvati in: {metadata_path}")

print("\n--- PIPELINE COMPLETATA CON SUCCESSO! ---")


--- [FASE 5/6] VALUTAZIONE MODELLO FINALE ---

Performance finale sul Test Set:
  - Loss: 0.9034
  - Accuracy: 0.7475

--- [FASE 6/6] SALVATAGGIO MODELLO E METADATI ---
Modello salvato in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_C_vector.keras
Metadati salvati in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_C_vector_metadata.json

--- PIPELINE COMPLETATA CON SUCCESSO! ---
